# 10 - Validación de Datos con Pandera
## Etapa de Calidad: Contratos de Datos en el Pipeline MLOps

**Objetivo:** Definir y aplicar contratos de datos con Pandera para garantizar  
que los datos que entran al pipeline (entrenamiento o inferencia) cumplen  
las expectativas antes de que el modelo los procese.

**¿Por qué validar datos?**  
Los modelos ML son especialmente sensibles a datos incorrectos:  
- Un campo nulo inesperado puede romper el pipeline en producción  
- Un rango fuera de lo esperado produce predicciones silenciosamente erróneas  
- Sin validación, los errores de datos se detectan tarde y son costosos

**¿Qué es Pandera?**  
Librería de validación de DataFrames para Python. Permite definir un **schema**  
(contrato de datos) que especifica tipos, rangos, nulabilidad y reglas de negocio.  
Si los datos no cumplen el contrato, Pandera lanza una excepción descriptiva.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import pandera as pa
from pandera import Column, DataFrameSchema, Check
from pandera.errors import SchemaError
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

# ── ROOT dinámico ──────────────────────────────────────────────────────────
ROOT = Path().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

DATA_DIR = ROOT / 'data' / 'processed'
RAW_DIR  = ROOT / 'data' / 'raw'

print(f'ROOT    : {ROOT}')
print(f'Pandera : {pa.__version__}')

## 1. Explorar los Datos que Vamos a Validar

Antes de definir el contrato, revisamos los rangos y estadísticas reales  
del dataset para establecer expectativas realistas.

In [ ]:
# ── Cargar datos raw (generados en NB01) ──────────────────────────────────
df_raw = pd.read_csv(RAW_DIR / 'housing_raw.csv')

print(f'Shape: {df_raw.shape}')
print(f'Columnas: {list(df_raw.columns)}')
print()
print('Estadísticas descriptivas:')
print(df_raw.describe().round(4).to_string())
print()
print('Tipos de datos:')
print(df_raw.dtypes)

## 2. Definir el Schema de Validación

Un **schema** en Pandera es un contrato formal que especifica:  
- **Tipo de dato** esperado para cada columna  
- **Rango de valores** aceptables (min, max)  
- **Nulabilidad**: si se permiten o no valores nulos  
- **Reglas de negocio**: checks personalizados con lógica de dominio

Este schema actúa como la primera línea de defensa del pipeline MLOps.

In [ ]:
# ── Schema de validación para datos raw ──────────────────────────────────
schema_raw = DataFrameSchema(
    columns={
        'MedInc': Column(
            dtype=float,
            checks=[
                Check.greater_than(0),
                Check.less_than_or_equal_to(20),
            ],
            nullable=False,
            description='Ingreso mediano del hogar (en decenas de miles)'
        ),
        'HouseAge': Column(
            dtype=float,
            checks=[
                Check.greater_than_or_equal_to(0),
                Check.less_than_or_equal_to(60),
            ],
            nullable=False,
            description='Edad media de las viviendas del bloque'
        ),
        'AveRooms': Column(
            dtype=float,
            checks=[
                Check.greater_than(0),
                Check.less_than_or_equal_to(100),
            ],
            nullable=False,
            description='Promedio de habitaciones por vivienda'
        ),
        'AveBedrms': Column(
            dtype=float,
            checks=[
                Check.greater_than(0),
                Check.less_than_or_equal_to(50),
            ],
            nullable=False,
            description='Promedio de dormitorios por vivienda'
        ),
        'Population': Column(
            dtype=float,
            checks=[
                Check.greater_than(0),
                Check.less_than_or_equal_to(40000),
            ],
            nullable=False,
            description='Población del bloque censal'
        ),
        'AveOccup': Column(
            dtype=float,
            checks=[
                Check.greater_than(0),
                Check.less_than_or_equal_to(20),
            ],
            nullable=False,
            description='Promedio de ocupantes por vivienda'
        ),
        'Latitude': Column(
            dtype=float,
            checks=[
                Check.in_range(32.0, 42.5),  # California: 32.5 a 42.0 aprox
            ],
            nullable=False,
            description='Latitud del centroide del bloque censal'
        ),
        'Longitude': Column(
            dtype=float,
            checks=[
                Check.in_range(-125.0, -114.0),  # California
            ],
            nullable=False,
            description='Longitud del centroide del bloque censal'
        ),
        'MedHouseVal': Column(
            dtype=float,
            checks=[
                Check.greater_than(0),
                Check.less_than_or_equal_to(5.5),
            ],
            nullable=False,
            description='Valor mediano de la vivienda (en $100K) — TARGET'
        ),
    },
    # Reglas a nivel de DataFrame
    checks=[
        # AveBedrms siempre debe ser <= AveRooms
        Check(lambda df: (df['AveBedrms'] <= df['AveRooms']).all(),
              error='AveBedrms no puede superar AveRooms'),
        # Sin duplicados
        Check(lambda df: not df.duplicated().any(),
              error='El dataset contiene registros duplicados'),
    ],
    name='schema_housing_raw',
    strict=False,   # permite columnas extra no definidas
    coerce=True,    # intenta convertir tipos automáticamente
)

print('Schema definido correctamente.')
print(f'Columnas validadas: {len(schema_raw.columns)}')
print(f'Checks a nivel DataFrame: 2')

## 3. Validar los Datos Reales

Aplicamos el schema al dataset real. Si pasa la validación, el pipeline  
puede continuar con confianza. Si falla, obtenemos un reporte detallado.

In [ ]:
# ── Validación 1: datos reales (deben pasar) ─────────────────────────────
print('Validando housing_raw.csv...')
try:
    df_validated = schema_raw.validate(df_raw, lazy=True)
    print('VALIDACION EXITOSA')
    print(f'Filas validadas: {len(df_validated):,}')
    print(f'Columnas:        {len(df_validated.columns)}')
    print(f'Nulos totales:   {df_validated.isnull().sum().sum()}')
except pa.errors.SchemaErrors as e:
    print('VALIDACION FALLIDA')
    print(e.failure_cases)

In [ ]:
# ── Validación 2: datos con errores inducidos (deben fallar) ─────────────
print('Creando dataset con errores deliberados...')

df_erroneo = df_raw.copy()

# Error 1: valor negativo en MedInc
df_erroneo.loc[0, 'MedInc'] = -1.5

# Error 2: latitud fuera de California
df_erroneo.loc[1, 'Latitude'] = 50.0

# Error 3: AveBedrms > AveRooms (imposible físicamente)
df_erroneo.loc[2, 'AveBedrms'] = df_erroneo.loc[2, 'AveRooms'] + 5

# Error 4: nulo en Population
df_erroneo.loc[3, 'Population'] = None

print('Errores introducidos:')
print(f'  Fila 0 MedInc     : {df_erroneo.loc[0, "MedInc"]} (esperado > 0)')
print(f'  Fila 1 Latitude   : {df_erroneo.loc[1, "Latitude"]} (esperado 32-42.5)')
print(f'  Fila 2 AveBedrms  : {df_erroneo.loc[2, "AveBedrms"]} > AveRooms {df_erroneo.loc[2, "AveRooms"]}')
print(f'  Fila 3 Population : {df_erroneo.loc[3, "Population"]} (no permite nulos)')
print()

try:
    schema_raw.validate(df_erroneo, lazy=True)
    print('INESPERADO: la validación pasó sin errores')
except pa.errors.SchemaErrors as e:
    print('VALIDACION FALLIDA (esperado)')
    print()
    print('Reporte de errores:')
    print(e.failure_cases[['schema_context', 'column', 'check', 'check_number',
                            'failure_case', 'index']].to_string(index=False))

## 4. Schema para Datos de Inferencia (Producción)

En producción, el modelo recibe datos de entrada sin el target `MedHouseVal`.  
Definimos un schema específico para validar las peticiones a la API  
antes de que lleguen al modelo.

In [ ]:
# ── Schema para inferencia (sin target) ──────────────────────────────────
schema_inferencia = DataFrameSchema(
    columns={
        'MedInc':    Column(float, checks=Check.greater_than(0), nullable=False),
        'HouseAge':  Column(float, checks=Check.in_range(0, 60),  nullable=False),
        'AveRooms':  Column(float, checks=Check.greater_than(0),  nullable=False),
        'AveBedrms': Column(float, checks=Check.greater_than(0),  nullable=False),
        'Population':Column(float, checks=Check.greater_than(0),  nullable=False),
        'AveOccup':  Column(float, checks=Check.greater_than(0),  nullable=False),
        'Latitude':  Column(float, checks=Check.in_range(32.0, 42.5), nullable=False),
        'Longitude': Column(float, checks=Check.in_range(-125.0, -114.0), nullable=False),
    },
    name='schema_inferencia',
    coerce=True,
)

# ── Simular peticiones a la API ────────────────────────────────────────────
peticiones = pd.DataFrame([
    # Válida: San Francisco
    {'MedInc': 8.5, 'HouseAge': 15, 'AveRooms': 6.0, 'AveBedrms': 1.2,
     'Population': 1200, 'AveOccup': 2.8, 'Latitude': 37.77, 'Longitude': -122.42},
    # Válida: Los Angeles
    {'MedInc': 6.2, 'HouseAge': 25, 'AveRooms': 5.5, 'AveBedrms': 1.1,
     'Population': 2000, 'AveOccup': 3.1, 'Latitude': 34.05, 'Longitude': -118.24},
    # INVÁLIDA: latitud de Madrid (error del cliente)
    {'MedInc': 5.0, 'HouseAge': 20, 'AveRooms': 4.0, 'AveBedrms': 1.0,
     'Population': 800,  'AveOccup': 2.5, 'Latitude': 40.42, 'Longitude': -3.70},
])

print('Validando peticiones de inferencia...')
for i, row in peticiones.iterrows():
    df_req = row.to_frame().T
    try:
        schema_inferencia.validate(df_req)
        print(f'  Petición {i+1}: VALIDA  (lat={row.Latitude:.2f}, lon={row.Longitude:.2f})')
    except SchemaError as e:
        print(f'  Petición {i+1}: INVALIDA → {e.args[0][:80]}')

## 5. Integración en el Pipeline MLOps

La validación de datos no es una herramienta aislada — debe estar integrada  
en dos puntos clave del pipeline:

In [ ]:
# ── Función de validación reutilizable para el pipeline ──────────────────
def validar_datos_entrada(df: pd.DataFrame,
                           modo: str = 'entrenamiento') -> pd.DataFrame:
    """
    Valida un DataFrame según el modo del pipeline.

    Parámetros
    ----------
    df   : DataFrame a validar
    modo : 'entrenamiento' (con target) o 'inferencia' (sin target)

    Retorna
    -------
    DataFrame validado (si pasa)

    Lanza
    -----
    SchemaErrors : si los datos no cumplen el contrato
    """
    schema = schema_raw if modo == 'entrenamiento' else schema_inferencia
    nombre = schema.name

    try:
        df_ok = schema.validate(df, lazy=True)
        print(f'[{nombre}] OK — {len(df_ok):,} filas validadas')
        return df_ok
    except pa.errors.SchemaErrors as e:
        n_errores = len(e.failure_cases)
        print(f'[{nombre}] FALLO — {n_errores} violaciones encontradas')
        print(e.failure_cases[['column', 'check', 'failure_case', 'index']]
              .head(10).to_string(index=False))
        raise


# ── Ejemplo de uso en el pipeline ─────────────────────────────────────────
print('=== Uso en entrenamiento (NB02) ===')
df_ok = validar_datos_entrada(df_raw, modo='entrenamiento')

print()
print('=== Uso en inferencia (API) ===')
peticion_valida = peticiones.iloc[[0]]   # Solo SF
df_inf = validar_datos_entrada(peticion_valida, modo='inferencia')

In [ ]:
# ── Visualización: resumen del contrato de datos ─────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))
ax.axis('off')
fig.suptitle('Contrato de Datos — Schema de Validación (Pandera)',
             fontsize=14, fontweight='bold')

columnas_info = [
    ('MedInc',     'float', '>0, ≤20',     'No', 'Ingreso mediano del hogar'),
    ('HouseAge',   'float', '0-60',         'No', 'Edad media viviendas del bloque'),
    ('AveRooms',   'float', '>0, ≤100',     'No', 'Promedio habitaciones/vivienda'),
    ('AveBedrms',  'float', '>0, ≤AveRooms','No', 'Promedio dormitorios/vivienda'),
    ('Population', 'float', '>0, ≤40000',   'No', 'Población del bloque censal'),
    ('AveOccup',   'float', '>0, ≤20',      'No', 'Promedio ocupantes/vivienda'),
    ('Latitude',   'float', '32-42.5',      'No', 'Latitud (California)'),
    ('Longitude',  'float', '-125 a -114',  'No', 'Longitud (California)'),
    ('MedHouseVal','float', '>0, ≤5.5',     'No', 'TARGET: precio mediano ($100K)'),
]

headers = ['Feature', 'Tipo', 'Rango', 'Nulos', 'Descripción']
col_w   = [0.12, 0.07, 0.13, 0.07, 0.45]
col_x   = [0.02]
for w in col_w[:-1]:
    col_x.append(col_x[-1] + w)

colors_h = ['#1F3864'] * 5
for j, (h, x, w) in enumerate(zip(headers, col_x, col_w)):
    ax.text(x, 0.95, h, transform=ax.transAxes,
            fontsize=10, fontweight='bold', color='white',
            bbox=dict(boxstyle='round', facecolor='#1F3864', alpha=0.9, pad=0.3))

for i, (feat, tipo, rango, nulos, desc) in enumerate(columnas_info):
    y = 0.85 - i * 0.085
    bg = '#F2F2F2' if i % 2 == 0 else 'white'
    valores = [feat, tipo, rango, nulos, desc]
    for j, (val, x, w) in enumerate(zip(valores, col_x, col_w)):
        color_text = '#1F3864' if j == 0 else '#262626'
        ax.text(x, y, val, transform=ax.transAxes,
                fontsize=9, color=color_text,
                fontweight='bold' if j == 0 else 'normal')

plt.tight_layout()
plt.show()

## 6. Conclusiones

### ¿Qué nos aporta la validación de datos en MLOps?

**Fallos silenciosos evitados:**  
Sin validación, un campo nulo o un rango incorrecto puede producir predicciones  
erróneas que el sistema acepta sin protestar. La validación convierte estos  
errores silenciosos en errores explícitos y descriptivos.

**Documentación viva del pipeline:**  
El schema de Pandera es a la vez código ejecutable y documentación del dato.  
No puede desactualizarse porque forma parte del pipeline.

**Cuándo ejecutar la validación en el ciclo MLOps:**

| Momento | Acción | Schema |
|---------|--------|--------|
| Ingesta de datos (NB01/NB02) | Validar antes de procesar | `schema_raw` |
| Antes de entrenar (NB03) | Validar train/test split | `schema_raw` |
| En la API (NB06) | Validar cada petición | `schema_inferencia` |
| En monitoreo (NB07) | Validar datos de producción | `schema_inferencia` |
| Tras re-entrenamiento | Validar nuevos datos | `schema_raw` |

**Pandera vs Great Expectations:**  
- **Pandera**: ligero, Pythónico, ideal para integrar en código de pipeline  
- **Great Expectations**: más robusto, genera reportes HTML, ideal para data engineering  
- Para un curso de MLOps con sklearn/FastAPI: Pandera es la opción más práctica